# Importing Libraries & Starting Kedro Context

In [ ]:
# To make use of kedro in eda

from kedro.framework.context import KedroContext
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project

import os

# Set up Kedro session
project_path = os.getcwd()
bootstrap_project(project_path)

session = KedroSession.create()
context = session.load_context()
catalog = context.catalog


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
import seaborn as sns
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd() / "src"))

# Import full modules (for reload)
import caie_nyp_batch3_mohammad_habib_410d.utils.etl as etl
import caie_nyp_batch3_mohammad_habib_410d.utils.viz as viz

import importlib
importlib.reload(etl)
importlib.reload(viz)

# Set custom plot style for consistency
viz.set_plot_style()

# Data Processing 

## Step 1 - Inspect Raw Data

In [ ]:
# Load dataset by catalog name
df = catalog.load("raw_bank_data")

# View the shape of df and first few data entries
print("Shape:", df.shape)
df.head()

In [ ]:
# View last few entries
df.tail()

## Step 2 - Standardize Column Names

In this step, after viualising the df, I realised that the column names were not standardized. So I cleaned them up to lowercase and removed spaces and replaced with "_".

In [ ]:
# Cleaning column names i.e. lowercase and _ for space

df = etl.step2_clean_column_names(df)

## Step 3 - Check Schema

One more thing I have noticed that is obviously wrong is the age column which had years, as in the units in the data entries. So to check if any more numeric/bool columns were secretly objects I will the schema of the df.

In [ ]:
# Info on the columns of df
df.info()

Almost all of the columns are objects.

## Step 4 - Detect Nulls & Duplicates

We will check of duplicates disregarding the client_id as it will ensure that all rows are unique. What I want to prevent is oversampling of patterns and as such I need to check the df without client_id.

In [ ]:
# Checking for null & dulplicate values
etl.null_duplicate_check(df, exclude_cols=['client_id'])

There are 5240 duplicate rows that have the same features & target label. About 10% of the data. This may lead to oversampled patterns and as such they will be dropped to prevent overfitting. A lot of null values in housing_loan.

In [ ]:
df = etl.step4_remove_duplicates(df)

## Step 5 - Check Target Distribution

Checking distribution of subscription status to guage how skewed it is.

In [ ]:
# Checking no. of subscribers
df['subscription_status'].value_counts()

In [ ]:
viz.plot_target_distribution(df,"subscription_status")

We can see that the dataset is heavily skewed to the not subscribed

## Step 6 - Fix Glaring Errors That Will Affect Further Analysis

Fixing years in age & use of 999 in previous contact days.

In [ ]:
# Fixing age column invalid values & use of 999 to signify no contact

df = etl.step6_fix_glaring_issues(df)

This step will allow me to continue further analysis of the numeric columns. As 999 was used as a placeholder in the previous_contact_days column. It would have affected the numeric properties of the column and as such greatly skewing the analysis. I intend to convert the column into bins and replace the nan's with no_contact effectively converting the column into a category but that will happen later. 


## Step 7 - Analyse Numeric Columns

After cleaning previous_contact_days column of 999 we can now analyse the numeric columns without skew.

In [ ]:
# Visualise the numeric columns
viz.plot_numeric_distribution(df.drop(columns=["client_id"], errors="ignore"))

In [ ]:
# Check distribution of numeric columns
df.describe()

- Age Max value is 150 which is impossible as humans do not live to 150 years old, Will review rows above 100

- Campaign Calls min value is in the negatives, Either this indicates the customer called the bank or invalid negative entry. Will review these negative cases 

- Campaign Calls max value is 56 which seems incorrectly high, to review these as well. (eill bin during modelling)

- Drastic increase of campaign calls between 75th percentile and max indicate there are high value outliers, likely due to focused outreach, will check if increased outreach causes subscription

- Drastic increase of previous contact days between 75th percentile and max indicate high value outliers, likely due to lack of outreach, will check if non recent outreach causes less subscriptions

## Step 8 - Analyze Categorical Columns

Most of the columns from the schema were objects. Lets analyse them to see the unique values & counts.

In [ ]:
# Lets check what are the different values for the object columns

etl.explore_object_columns(df)

 - None and Unknown in many columns

 - Same semantic meaning categories in contact method

## Step 9 - Fix Easily Identifiable Issues

Fixing multiple categories with same semantic value in contact method. Cleaning (admin. --> admin).

In [ ]:
# Fix occupation, contact method & loan columns
df = etl.step9_clean_categoricals(df)

## Step 10 - Impute Data

Education level is often times a good indicator of the kinds of jobs a person has. To impute the remaining unknown occupations with a value, we will use the most occuring occupation for each rows education level. There should be no cases where a row has >1 unknown data entry and as such this is able to be done with any issues.

In [ ]:
df["occupation"].value_counts(dropna=False)


In [ ]:
df = etl.step10_impute_unknown_occupation_by_education(df)

In [ ]:
df["occupation"].value_counts(dropna=False)

## Step 11 - Drop Rows With Missing/Invalid Data

I will assume that none and unknown mean the same. This is because unknown implies the data entry was not given/recieved and none implies that data is missing. Either way, they signify a very similar situation and as such I will take them to mean the same thing. As such I will not flag none as true none etc etc and just take them as the same. This will then be applied to the loan columns & credit_default that have both unknown and none.

In [ ]:
# Run Null & Duplicate Check again
etl.null_duplicate_check(df)

In [ ]:
# Removing all unknowns & none as well as removing age>100. Ignore previous_contact_days as nan means no contact & ignore housing_loan as I will handle it seperately as it has >50% missing
df = etl.step11_drop_incomplete_invalid_rows(df)

In [ ]:
# Counting no. of unknowns 
etl.count_unknowns_per_column(df)

In [ ]:
# Run Null & Duplicate Check again
etl.null_duplicate_check(df)

# Checking no. of subscribers to ensure that signigicant portion of True were not removed
df['subscription_status'].value_counts()

## Step 12 - Fix Data Types

 Converting the object datatypes to category datatypes as there are distinct seperation between the strings and as such should be in category datatype. Converting credit_default, personal_loan & previous contact days to boolean.

In [ ]:
# Lets check what are the different values for the object columns

etl.explore_object_columns(df)

In [ ]:
# Review Data Types one last time before proceeding with data type changes
df.info()

In [ ]:
# Run df through datatype conversion function
df = etl.step12_convert_datatypes(df)

In [ ]:
# Verifying the changes to ensure no unecessary/unwanted outcomes occured

cols_to_check = ["credit_default", "personal_loan", "subscription_status"]

for col in cols_to_check:
    print(f"\n{col} value counts:")
    print(df[col].value_counts(dropna=False))


In [ ]:
# Verify Schema
df.info()

## Step 13 - Handling campaign_calls

In [ ]:
# Check if there are any rows with 0 calls to verify a suspicion

print("Clients with 0 calls:", (df["campaign_calls"] == 0).sum())
print("\nClients with -ve calls:", (df["campaign_calls"] < 0).sum())

This clearly shows that all clients with valid data were at least called once. As such, I will assume that all clients were called more than once. As a result, any -ve values for calls do not align with my assumption. As there is no possible way to know how many times these clients were called, we have to drop these rows as any method of imputing will skew the models understanding of the data.

In [ ]:
# Dropping rows with calls < 0
df = etl.step13_drop_negative_campaign_calls(df)

In [ ]:
# Checking no. of subscribers
df['subscription_status'].value_counts()

## Step 14 Addressning Rows With Unknowns & Improbable or Illogial Cases

### Step 14: Flag Columns With Unknowns & Improbable or Illogical Cases

Impossible or Improbable Cases

- Credit default = yes & loan present represent an anomalous occurrence.

- Married below age 18

- Illiterate but have one of the following occupations: admin, technician, management, entrepreneur, self-employed, student. Since I am assuming education is a good indication of what a persons job is, I have to ensure my assumption is present throughout.

- Age ≥ 55 but occupation = student

In [ ]:
# Checking for these situations
etl.check_improbable_cases_exist(df)

I will drop these 3 rows to ensure my model is following my assumptions.

In [ ]:
# Dropping niche rows
df = etl.step14_remove_improbable_cases(df)

## Step 15 - Correlations Between Columns

In [ ]:
# Checking colinearity between feature columns
etl.check_colinear_columns(df, threshold=0.7, exclude=["client_id"])

We can see feature columns are not colinear and as such provide new information. Therefore there is no need to drop/combine features.

## Step 16 - Outlier Handling

In [ ]:
# Visualise the numeric columns after all data cleaning
viz.plot_numeric_distribution(df.drop(columns=["client_id"], errors="ignore"))

In [ ]:
df.info()

In [ ]:
# Check distribution of numeric columns
df.describe()

We can see there are some very large value outliers remaining. However, they are valid outliers and should not be removed. What I will do to aid in later steps is bin the numeric ages for ease of visualisation. We will bin the columns and convert it into a category as well. What this will do to previous_contact_days is allow the data to be fed into a model as nulls will be converted to "no_contact" and as such be represented by their own category. The rest of the numeric columns will be binned in the small buckets to preserve as much information.

In [ ]:
# Binning numeric columns
df = etl.step16_bin_numeric_features(df)

## Step 17 - Feature Selection

We will check the list of columns remaining.

In [ ]:
print(df.columns.tolist())

We will select all columns to be fed into the models except for previous_contact_days due to it having nulls which is why it was encoded into another column as well as credit_default as it only has 2 true values and the rest are negative and as such they will cause noise instead of provide information due to small sample size. client_id will be seperated during the training process as it does not provide any information on the target other than being a unique identifier for the rows.

In [ ]:
# Check model input df
df = etl.pass_to_model_inputs(df)

In [ ]:
df.head()

In [ ]:
df.info()

## Step 18 - Review Changes & Audit

Lets review the final dataset that will be used for model training as well as visualise the changes made to it.

In [ ]:
# Use the function to specify the df & the target label
etl.audit_dataset(df, target="subscription_status")


# Derviving Insights from Processed Data For Feature Engineering

Using final 3 methods of data analysis to extract information from data

Descriptive (Done in above section) -> Diagnostic -> Predictive -> Presprictive

These patterns will be used to gain business insights into the data, these insights will then be used to create models with feature engineered data which should perform a lot better on TRUE class.

## Descriptive 


### Age vs Subscription Status

In [ ]:
sns.histplot(data=df, x="age_group", hue="subscription_status", multiple="stack", bins=20)
plt.title("Age distribution by subscription status")
plt.show()

In [ ]:
# Rate of subsciption based off age group
etl.analyze_feature_ratio(df, 'age_group')

Key Observations to take away

- Older Customers seem more likely to be subscribed, even with a smaller sample size, we can see that above the age of 59 across 2 age bins, the split between subscribed and non subscribed customers is abouut 50/50. Suggesting the bank should target these individuals more.

- Anomaly for ages below 20 where the split is about even as well however due to very small sample size conclusion is hard to make. May be due to outliers or other factors.

- Across other ages 30s-60s we can see the false to true ratio is quite high and this suggests working adults are less likely to be subscribed to the term deposit.

- Amongst the younger clients, we can see young adults have the best reception to the term deposits

### Occupation vs Subscription Status

In [ ]:
viz.plot_categorical_distributions(
    df=df,
    columns=["occupation"],
    hue="subscription_status"
)

In [ ]:
# Rate of subsciption based off occupation
etl.analyze_feature_ratio(df, 'occupation')

Key takeaways

- Students are the most receptive to term deposits, followed by retired individuals.

- Across the rest of the jobs we can see a general trend where as job prestige decreases and likly income the subscription rate decreases.

- Key outliers being unemployed and entrepreneurs. Where unemployed people have a 14.6% subscription rate & entrepreneurs who likely invest money elsewhere have a 9.2% subscription rate.

### Education vs Subscribed Status

In [ ]:
# Visualize education level vs subscription
viz.plot_categorical_distributions(
    df=df,
    columns=["education_level"],
    hue="subscription_status"
)

In [ ]:
# Rate of subsciption based off occupation
etl.analyze_feature_ratio(df, 'education_level')

Key Takeaways

- General trend is as a clients education level increases, the subscription rate increases with the highest realistic being 15.9% for university degree holders & the lowest being basic.6y at 8.7%.

- Clear outlier being illiterate which has too small of a sample size to gather an accurate analysis. Another outlier is unknown, the issue being that as unknown carries no semantic meaning other than missing it is not valid to determine that not knowing the education level is a good factor.

### Maritial Status vs Subscription Status

In [ ]:
viz.plot_categorical_distributions(
    df=df,
    columns=["marital_status"],
    hue="subscription_status"
)

In [ ]:
# Rate of subsciption based off occupation
etl.analyze_feature_ratio(df, 'marital_status')

Key Takeaway

- Single individuals seem to have the highest subscription rate than all other categories at ~15.9% may be due to more disposable income to put away.

- Unknown is an invalid outlier as there is too small a sample size to make an accurate deduction.

### Personal Loan vs Subscription Status

In [ ]:
viz.plot_categorical_distributions(
    df=df,
    columns=["personal_loan"],
    hue="subscription_status"
)

In [ ]:
# Rate of subsciption based off occupation
etl.analyze_feature_ratio(df, 'personal_loan')

Key Takeaway

- Disregarding unknown as it contains no semantic information, we can see that those with no loans are more likely to subscribe than those with loans. This is likely due to the fact these clients with no loans have no financial dependency on the bank (needing to pay the loan off)

### Contact Method vs Subscription Status

In [ ]:
viz.plot_categorical_distributions(
    df=df,
    columns=["contact_method"],
    hue="subscription_status"
)

In [ ]:
# Rate of subsciption based off occupation
etl.analyze_feature_ratio(df, 'contact_method')

Key takeaways

- Cellular contact is much more effective, it has a higher rate of subscription than telephone;17% as compared to around 6% which is 3x more effective and is a clear indicator that the outreach method is a key factor

### Campaign Calls vs Subscription Status

In [ ]:
sns.histplot(data=df, x="campaign_call_bin", hue="subscription_status", bins=30, multiple="stack")
plt.title("Campaign Calls by Subscription Status")
plt.show()


In [ ]:
# Rate of subsciption based off occupation
etl.analyze_feature_ratio(df, 'campaign_call_bin')

Key Takeaways

- Highest conversion rate 15.7% when only contacted once

- Conversion rate decreases as no. of calls increase likely signifying annoyonace or fatigue from the number of calls.

### Previous Contact Days vs Subscribed Status

In [ ]:
viz.plot_categorical_distributions(
    df = df[df["prev_contact_bin"] != "No Contact"].copy(),
    columns=["prev_contact_bin"],
    hue="subscription_status"
)

In [ ]:
# Rate of subsciption based off occupation
etl.analyze_feature_ratio(df, 'prev_contact_bin')

Key Takeaways

- Recent Contact within 1 week leads to higher subscription rate.

- Subscription rate declines as contact days between customer and bank increases

- We can see those that were contacted and the bank had a conversation with. Led to extremely high subscription rates which indicates that the bank should focus on contacting and speaking with customers more often

## Verification of Trends

In [ ]:
from itertools import product

# Best bins/categories from EDA
rules = {
    "age_group": ["61-70","71-80", "80+"],
    "occupation": ["student", "retired", "admin"],
    "education_level": ["university.degree", "professional.course", "high.school"],
    "marital_status": ["single"],
    "contact_method": ["cellular"],
    "campaign_call_bin": ["1", "2"],
    "prev_contact_bin": ["0-3", "4-7"]
}

In [ ]:
from itertools import combinations

# You can change this to 3 for triple combinations
combi_size = 2

# Generate all unique combinations of the features
feature_combos = list(combinations(rules.keys(), combi_size))


In [ ]:
results = []

for features in feature_combos:
    # Get all value combinations across these features
    value_combos = list(product(*[rules[f] for f in features]))
    
    for values in value_combos:
        condition = np.ones(len(df), dtype=bool)
        for f, v in zip(features, values):
            condition &= (df[f] == v)
        
        subset = df[condition]
        n = len(subset)
        
        if n > 20:  # ignore tiny sample sizes
            true_rate = subset["subscription_status"].mean()
            results.append({
                "features": features,
                "values": values,
                "n": n,
                "true_rate": round(true_rate, 3)
            })


In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="true_rate", ascending=False).reset_index(drop=True)
results_df.head(5)

These combinations are mostly dominated by prev_contact_bin, lets cut off the combinations at 3x the true_rate than the overall dataset and then inspect the combinations from there.

In [ ]:
# Overall true rate
base_true_rate = df["subscription_status"].mean()  # ~0.15
print(base_true_rate)
# 3x better than base
min_acceptable_rate = base_true_rate * 2  # ~0.29
print(min_acceptable_rate)

In [ ]:
# Filter out rules to only keep those that add value and remove those that might cause noise
filtered_rules = results_df[
    (results_df["true_rate"] >= min_acceptable_rate) &
    (results_df["n"] >= 30)
].copy()

filtered_rules = filtered_rules.sort_values("true_rate", ascending=False).reset_index(drop=True)
filtered_rules


Clearly combinations of these features do lead to higher than normal true rate. We will label these as weak positives in the feature engineering pipeline so the model has an indicator for higher likelyhood true cases.

# Model Performance

## Start Kedro Context

In [ ]:
from kedro.framework.context import KedroContext
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay
import seaborn as sns
import json

# Load context
from kedro.framework.session import KedroSession

with KedroSession.create() as session:
    context = session.load_context()
    catalog = context.catalog

## Base Models

### Logistic Regression (Pre Feature Engineering)


In [ ]:
# Load metrics & predictions
logistic_metrics = catalog.load("logistic_model_metrics")
logistic_predictions = catalog.load("logistic_predictions_test")
# Display metrics & plot
viz.plot_classification_evaluation(logistic_metrics, logistic_predictions, model_name="Logistic Regression (Pre Feature Engineering)")

Overall accuracy is valid at ~84%. However, according to task at hand main objective is to identify the most amount of potential subscribers. As such this models performance is not addressing the objective. We will first attempt to feature engineer columns for model to learn more nuanced information to increase accuracy on class 1.

### Lightgbm (Pre Feature Engineering)

In [ ]:
# Load metrics & predictions
lightgbm_metrics = catalog.load("lightgbm_model_metrics")
lightgbm_predictions = catalog.load("lightgbm_predictions_test")
# Display metrics & plot
viz.plot_classification_evaluation(lightgbm_metrics, lightgbm_predictions, model_name="Lightgbm (Pre Feature Engineering)")

### xgboost (Pre Feature Engineering)

In [ ]:
# Load outputs
xgb_metrics = catalog.load("xgboost_model_metrics")
xgb_predictions = catalog.load("xgboost_predictions_test")

# Visualize
viz.plot_classification_evaluation(xgb_metrics, xgb_predictions, model_name="xgboost (Pre Feature Engineering)")


### Pre Semi-Supervised Learning Summary

Model performance across the board is poor for class 1. Overall accuracy is good however, for our intended task of identifying potential subscribers it performs extremely poorly. To address this, we will start with semi supervised learning using the trends & business insights learned from analysing the data. This should help the model be able to identify the true class much better.

## Models With feature Engineering

### ENG-Logistic Regression

In [ ]:
# Load metrics and predictions for eng models
eng_logistic_metrics = catalog.load("eng_logistic_metrics")
eng_logistic_predictions = catalog.load("eng_logistic_predictions")

# Display metrics & plot
viz.plot_classification_evaluation(eng_logistic_metrics, eng_logistic_predictions, model_name="ENG-Logistic Regression")

### ENG-Lightgbm

In [ ]:
# Load metrics and predictions for eng models
eng_lgbm_metrics = catalog.load("eng_lgbm_metrics")
eng_lgbm_predictions = catalog.load("eng_lgbm_predictions")

# Display metrics & plot
viz.plot_classification_evaluation(eng_lgbm_metrics, eng_lgbm_predictions, model_name="ENG-Lightbgm")

### ENG-xgboost

In [ ]:
# Load metrics and predictions for eng models
eng_xgb_metrics = catalog.load("eng_xgb_metrics")
eng_xgb_predictions = catalog.load("eng_xgb_predictions")

# Display metrics & plot
viz.plot_classification_evaluation(eng_xgb_metrics, eng_xgb_predictions, model_name="ENG-xgboost")

### ENG Summary

This is a clear case of data limitation. There seems to be very little I can do to improve the overall performance of the TRUE class. Even with columns that add information on weak_positives, all models cannot pick up the nuances to correctly and accuractly identify all TRUE cases. As such, my model tuning will focus on the recall metric. The situation demands that we identify potential subscribers and as such focusing on recall will allow us to capture all likely subscribers but trade off with accuracy. As a result, the marketing team will end up contacting many more non subscribers but they will reach out to many more potential subscribers than if I were to focus on any other metric, thus able to secure more fixed deposits than if they did not. Moving forward, I will tune the eng models to achieve the best recall and get them to around 0.8 recall which may lead to other metrics being very poor for TRUE class but this is done consciously as a valid tradeoff. 

## Tuned Models

### Tuned ENG-Logistic

In [ ]:
# Load metrics and predictions for Tuned logistic model
tuned_logistic_metrics = catalog.load("tuned_logistic_metrics")
tuned_logistic_predictions = catalog.load("tuned_logistic_predictions")

# Display metrics & plot
viz.plot_classification_evaluation(tuned_logistic_metrics, tuned_logistic_predictions, model_name="Tuned ENG-Logistic")

Methods used to fine tune

- Train Test split using stratification, preserves the proportion of classes in train and test

- Oversampling with SMOTE to aid with recall of class true. Reduces bias of model towards class 0

- Grid search for hyper parameters.

- Threshold tuning to find the best one that satisfies recall_class 1 >= 0.7 & precision_class_1 >= 0.2 to prevent too many falsepositives.

### Tuned ENG-lightgbm

In [ ]:
# Load metrics and predictions for Tuned logistic model
tuned_lgbm_metrics = catalog.load("tuned_lgbm_metrics")
tuned_lgbm_predictions = catalog.load("tuned_lgbm_predictions")

# Display metrics & plot
viz.plot_classification_evaluation(tuned_lgbm_metrics, tuned_lgbm_predictions, model_name="Tuned ENG-Lightgbm")

### Tuned  ENG-xgboost


In [ ]:
# Load metrics and predictions for Tuned logistic model
tuned_eng_xgb_model_metrics = catalog.load("tuned_eng_xgb_model_metrics")
tuned_eng_xgb_model_predictions = catalog.load("tuned_eng_xgb_model_predictions")

# Display metrics & plot
viz.plot_classification_evaluation(tuned_eng_xgb_model_metrics, tuned_eng_xgb_model_predictions, model_name="Tuned ENG-xgboost")

### Summary on Tuned models

After initial modeling, all three models—Logistic Regression, LightGBM, and XGBoost—were fine-tuned with a focus on improving recall. Techniques such as SMOTE oversampling, threshold tuning, class weighting, and hyperparameter search were applied. All 3 models performed better on the data and were able to capture a much higher % of true instances as compared to pretuning. The tuned models showed consistent gains in identifying the positive class while maintaining acceptable overall performance. These models are now ready for comparison, evaluation, and potential deployment